# Chapter 4

So far we have worked with data which has only been filtered using band-pass filtering which helps in getting rid of a lots of noises generated from power line interference, however, there are still noises due to eye-blink (EOG*), ECG artefacts that still needs to be addressed. <br />
Therefore in this chapter we will look into how to remove artefacts from our epoched data.

> *In EEG analysis, EOG refers to the electrical activity generated by eye movements, including:
> +   Blinks
> +   Vertical eye movements
> +   Horizontal eye movements
>
> These activities create large voltage changes because the eye behaves like an electrical dipole (cornea is positive, retina is negative). When the eyes move, this dipole shifts, and EEG electrodes pick up that change as artefacts.

## Libraries & Config

In [ ]:
import pathlib
import matplotlib

import mne

matplotlib.use("QtAgg")
mne.set_log_level("warning") # this will make sure only warnings and errors are printed

In [ ]:
import mne_bids

## Read Epoch data

In [ ]:
epochs = mne.read_epochs(
    pathlib.Path("out_data") / "epochs-epo.fif"
)

epochs

> When we created our epochs, we applied a baseline correction of (None, 0), meaning we subtracted the mean from the start of the epoch up to the event onset. <br />
Sometimes we may want to use a different baseline. While MNE does allow calling `.apply_baseline()` again, this is not the preferred method, because the data have already been baseline-corrected once. Instead, it is better to re-create the epochs from the uncorrected data and apply the desired baseline only once.

In [ ]:
epochs.plot()

## Reject artifacts based on channel signal amplitude

Now we would like to drop epochs based on channel amplitude because excessively large amplitudes almost always indicate artefacts rather than real brain activity. <br />
We can perform such task by first defining two dictionaries:
1. **reject criteria** - which specifies if in an epoch we have any corresponding channels with similar or more than specified amplitude drop that epoch.
2. **flat criteria** - which specifies if in an epoch we have any corresponding channels with amplitude below than the specified amplitude drop that epoch (because that channel is too flat).

In [ ]:
reject_criteria = dict(
    mag=3000e-15, # 3000 fT
    grad=3000e-13, # 3000 fT/cm
    eeg=150e-6, # 150 µV microvolts
    eog=200e-6, # 200 µV
)

flat_criteria = dict(
    mag=1e-15, # 1 fT
    grad=1e-13, # 1 fT/cm
    eeg=1e-6 # 1 µV
)

Now to drop the bad epochs we can simply call the `.drop_bad()` function of the `Epochs` instance and pass the reject and flat criteria:

In [ ]:
artifacts_rejected_epochs = epochs.copy().drop_bad(reject=reject_criteria, flat=flat_criteria)

Originally we had `320` events and after dropping the bad epochs we now have `275`. Now that is quite a number of events that were dropped now to get what channels epochs get's dropped the most we can call the `.plot_drop_log()` method of the `Epochs` instance: 

In [ ]:
artifacts_rejected_epochs.plot_drop_log()

From the plot we can see the channel that had the most drops was `EOG`and few of `EEG` which we can speculate to that this could be realted to the ocular activity.

> In EEG, "ocular activity" refers to electrical signals from eye movements (blinking, rolling) that create large, distinct artifacts, primarily in frontal leads, because the eye acts as a positive corneal/negative retinal dipole; these artifacts are crucial to identify and often remove (artifact rejection/correction) as they easily overwhelm genuine brainwave signals, though eye movement itself can also track cognitive processes. 


So, let's plot the ERP image of the `visual` events to see if the data is more clear now:

In [ ]:
artifacts_rejected_epochs['Visual'].plot_image()

We still cannot see any clear picture from the ERP image of all combined images maybe we should look at the channel which is closer to the `primary visual cortex` which is at the back of the head. <br />
To find all the sensors locations we can call the `.plot_sensors()` function of the `Epochs` instance and pass the channel type we want to look into:

In [ ]:
artifacts_rejected_epochs.plot_sensors(ch_type='eeg')

From the plot we can see the channel/sensor `EEG 060` is the sensor located at the back of the head. So, now let generate the ERP image of that channel:

In [ ]:
artifacts_rejected_epochs['Visual'].plot_image(picks='EEG 060')

All together, after applying the rejection criteria, we found that most dropped epochs were due to EOG activity and a few EEG channels with large amplitudes, indicating eye blinks and other artefacts. Removing these noisy epochs improved the signal quality.

> When we then plotted an ERP image from an occipital EEG channel (over the visual cortex), we observed a clear change in brain activity around 0.2 seconds after stimulus onset. This timing is consistent with later stages of visual processing, where the brain is interpreting and processing the visual information rather than just detecting the light.

## Source Separation/ Artifact correction (or artifact removal) techniques

We initially reject epochs based on channel amplitude because unusually large amplitudes are almost always caused by artefacts such as eye blinks, muscle activity, or sudden movements, rather than real brain signals. Removing these epochs helps improve overall data quality and this method is known as **Artifact rejection technique**.

However, if many epochs are rejected this way, we lose a large number of trials, which can reduce the reliability of the results. This is especially problematic when only a small part of an epoch (for example, an eye blink) is responsible for the large amplitude.

To address this, methods such as **SSP (Signal Space Projection)** and **ICA (Independent Component Analysis)** are used. Instead of discarding entire epochs, these methods aim to remove the artefact itself while preserving the rest of the brain signal, and these are called **Artifact correction techniques**.

*SSP* works by identifying patterns in the data that are characteristic of specific artefacts and projecting these patterns out of the signal. *ICA*, on the other hand, separates the recorded signal into independent components corresponding to different sources, such as brain activity, eye blinks, or heartbeats, allowing the artefact-related components to be removed selectively.

Therefore, using *SSP* or *ICA* allows us to retain more trials while still reducing artefacts, achieving a better balance between data quality and data quantity.

> Note: *SSP* is popular for *MEG* artifact correction, but can still be applied to *EEG* data 

## Signal Space Projection (SSP)

To get a better understanding of SSP, we can think of SSP working like a pair of polarised sunglasses that filter out specific "direction" of light (noise) while letting the rest through.<br />
Imagine you are looking at a beautiful landscape (your brain data), but there is a blinding glare from the sun (the heartbeat/ECG artifact).
+ **Without sunglasses**: The glare is everywhere, washing out the details of the landscape.
+ **With sunglasses**: The lenses are designed to block light coming from one specific angle (the glare) while letting light from all other angles (the landscape) pass through.

So, SSP mathematically finds the angle/subspace/vector that the heartbeat noise lives in and creates a filter to block exactly that angle.

Luckily *MNE* comes with some convenient functions, which automatically helps us detech *EOG* and *EEG* artefacts, creates *SSP* projectors to remove these artefacts from the data. Now, to perform *SSP* we have follow the following steps

1. Read the raw BIDS data because to create the SSP projectors we need the raw data:

In [ ]:
bids_root = pathlib.Path("out_data/sample_BIDS")

bids_path = mne_bids.BIDSPath(
    subject='01',
    session='01',
    task='audiovisual',
    run='01',
    datatype='meg',
    root=bids_root
)

raw_bids = mne_bids.read_raw_bids(bids_path=bids_path)

# load the data in memory
raw_bids.load_data()
# perform band-pass filtering
raw_bids.filter(
    l_freq=0.1,
    h_freq=40.0
)

2. Now we will use the `compute_proj_ecg()` and `compute_proj_eog()` of the `mne.preprocessing` module to create the projectors, respectively they try to find and correct the artefacts of the **ECG** and **EOG**, and for it, these functions needs to know how many "lenses" to build for each type of sensor and that is defined by these parameters `n_mag`, `n_grad` and `n_eeg`. These parameters tells *MNE* how many *noise directions/angle* to block for each sensor type, for example in the case of *ECG*: 
+ `n_mag = 1`: means, find the 1 strongest direction of heartbeat noise in the *Magnetometers* and block it.
+ `n_grad = 1`: means, find the 1 strongest direction of heartbeat noise in the *Gradiometers* and block it.
+ `n_eeg = 0`: means, do not try to block anything in the EEG channels and it could be because the heart noise in our EEG sensors is small enough that we can ignore it (or fix it later with ICA) and in that case this will save computational power and avoids "over-cleaning" the EEG if it's not strictly necessary.

> **Why not just use 1 for everything?** <br/>
_Different sensors "see" noise differently. Magnetometers (MAG) measure the magnetic field directly and often pick up deep, distant noise (like the heart) very strongly. Gradiometers (GRAD) measure the difference between two nearby points and are naturally better at ignoring distant noise. You might need a stronger filter (more projectors) for MAGs than for GRADs, or vice versa._

Furthermore, we also need to pass another parameter `average=`, this is a critical trick to make the "sunglasses" precise:
+ `average=False`: The algorithm looks at every single heartbeat individually. This is noisy; one weird heartbeat might throw off the calculation.
+ `average=True`: MNE takes all the heartbeats it found, stacks them on top of each other, and computes an average heartbeat artifact.

> With `average=True`, it’s like taking 100 photos of a ghost to see exactly what it looks like, rather than trusting just one blurry photo.<br />
So, the "sunglasses" are then built to block this averaged, clean representation of the noise. This ensures you block the heartbeat itself and not random brain activity that happened to occur at the same time.

Now, as we know we don't have any *ECG* channel in our data, but luckily we can use our *magnetometers (MAG)* channels to get a pretty clear signal of the *ECG*, and that is because since the heart is a giant electrical generator, its magnetic field is strong enough to be picked up by the sensitive brain scanners (MAGs) located near the chest/neck. MNE reconstructs a "virtual" ECG channel by looking for the regular "thump-thump" pattern in the magnetometer data.


In [ ]:
ecg_projs, ecg_events = mne.preprocessing.compute_proj_ecg(
    raw=raw_bids,
    n_grad=1,
    n_mag=1,
    n_eeg=0,
    average=True,
)

eog_projs, eog_events = mne.preprocessing.compute_proj_eog(
    raw=raw_bids,
    n_grad=1,
    n_mag=1,
    n_eeg=1,
    average=True,
)

In [ ]:
ecg_projs

In [ ]:
eog_projs

Now after we have created the projectors, we can see both *ECG* and *EOG* projectors have 3 similar *PCA-v1*, *PCA-v2* and *PCA-v3* projectors and that is because these projectors were actually already included in the raw data from the get-go. Separately, the *ECG* projectors have two additional projectors in them *ECG-planar*, *ECG-axial* one for the 'gradiometers' and one for the 'magnetometers', respectively; the *EOG* projectors have one additional projector compared to *ECG*, which is *EOG-eeg* which is for the *EEG* channels.

Now we are going to combine these projectors, to have one single list of all the projectors:

In [ ]:
projs = eog_projs + ecg_projs

projs

Now let's add these list of projectors to our epochs calling the `.add_proj()` function of the *Epochs* instance.<br />
Then we can then plot the epochs and press `j` to toggle on and off the masking of the projectors:

In [ ]:
epochs.add_proj(projs)

epochs.plot()

Now to apply the projections on our epochs we can call the `.apply_proj()` function of the `Epochs` instance:

In [ ]:
ssp_epochs = epochs.copy().apply_proj()

Let's plot the ERP image for the visual events to see if there are any improvements:

In [ ]:
ssp_epochs['Visual'].plot_image()

Let's look at the electrode which is at the back of the head and also for the one which is at the front of the head:

In [ ]:
ssp_epochs['Visual'].plot_image(picks='EEG 060')

In [ ]:
ssp_epochs['Visual'].plot_image(picks='EEG 002')

## Independent Component Analysis (ICA)

ICA takes your mixedup sensor data, separates it into distinct/independent "sources" (heart, eyes, brain), lets you delete the bad ones, and then mixes it back together into a clean signal:
1. **Un-mixing:** You feed your raw data into ICA. *It assumes that the "sources" are statistically independent (the heart doesn't beat because you blinked)*.
2.  **Components:** It returns a list of "Independent Components" (ICs).
3. **Visual Check:** You look at these components.
- Is it a heartbeat? (Looks like a rhythmic pulse).
- Is it a blink? (Looks like a huge spike at thfront).
- Is it brain signal? (Looks like aa wavy alphoscillation).
4. **Rejection:** You mark the "Heartbeat" and "Blink" components as "BAD".
5. **Reconstruction:** ICA rebuilds your original data without those bad components. You now have clean data!

So, ICA can remove artifacts that "overlap" with your brain signal in space, as long as they have different timing.

<div style="text-align: center;">
<img src="imgs/brain_waves.png" alt="Brain Waves" width="800">
<br />
*Diagram by: <a href="https://psychedelicreview.com/altered-oscillations-the-modulatory-effect-of-dmt-on-brain-waves/">psychedelicreview</a>
</div>


Now, to perform *ICA* we have follow the following steps

1. As *ICA* tends to perform best when the epochs comes with a *higher cut-off fequency filter (more like a high-pass frequncy of 1 Hz and not 0.1 Hz unlike before)*. Therefore, we have to read the raw BIDS data again to create the epochs with the right filters:

In [ ]:
bids_root = pathlib.Path("out_data/sample_BIDS")

bids_path = mne_bids.BIDSPath(
    subject="01",
    session="01",
    task="audiovisual",
    run="01",
    datatype="meg",
    root=bids_root
)

raw_bids = mne_bids.read_raw_bids(
    bids_path=bids_path
)

In [ ]:
# load the whole data into memory
raw_bids.load_data()
# apply band pass filter with a higher high-pass filter
raw_bids.filter(
    l_freq=1,
    h_freq=40
)

2. Now, before creating epochs of the raw data, we have to perform another step where depending on before starting the ICA steps (i.e., reading the raw data again and performing a higher high-pass filter) when we created a orig epochs if we applied a rejection procedure to remove specific epochs, in that case we would like to extract the ones that were kept (in our case, all, because  we didn't apply any rejection procedure before saving the epochs in [chapter 3](/chapter_3.ipynb); but this could be different in a real-world scenario, and you want to calculate ICA on the same set of epochs you're actually feeding into your analysis!). <br />
Luckly, `Epochs` instances have a `.selection` attribute to tell us which epochs were kept and then using the kept epochs we can exract their *events* to later use it when creating the new epochs for ICA:

In [ ]:
# read the original epochs
orig_epochs = mne.read_epochs(
    pathlib.Path("out_data") / "epochs-epo.fif"
)
# extract the kept epochs info
orig_epochs_selection = orig_epochs.selection
orig_epochs_selection

In [ ]:
# read all the event of the newly read raw data
events, event_id = mne.events_from_annotations(raw_bids)
# only select the events of the kept on orig epochs
events = events[orig_epochs_selection]

3. Let's now create the new epochs using the subset of events extracted from kept original epochs. This way we will only the keep epochs we intended to keep (in this case it is all of them):

In [ ]:
# as we would like to retain the epochs timing and basline info
tmin = orig_epochs.tmin
tmax = orig_epochs.tmax
baseline = orig_epochs.baseline

epochs_ica = mne.Epochs(
    raw=raw_bids,
    events=events, # subset events
    event_id=event_id,
    tmin=tmin,
    tmax=tmax,
    baseline=baseline,
    preload=True
)

In [ ]:
epochs_ica.info

4. Now let's fit the epochs to ICA to get the independet components. Luckily, *MNE* has `mne.preprocessing.ICA()` class and it takes a number of parameters:
+ `n_components`: It tells ICA how many "independent sources" to find your data. It works in two ways:
    -   Interger (e.g., `20`): Find exactly 20 sources.
    -   Float (e.g., `0.999`): Find as many sources as needed to explain 99.9% of the variance.
    > It is recommended to use `0.999` (nearly all variance) or a fixd interger like `15` or `20` for EEG and `40+` for MEG, because setting it to even as low as `0.8` means you are throwing away 20% of your data's information before ICA even starts. This is usually too aggressive because you want to give ICA enough "room" to separate the noise from the brain signals.
+ `method='picard'`: It tells which specific algorithm to use to solve the ICA equation and there are three type of it:
    - `fastica`: The classic default. Fast, but sometimes less accurate on difficult data.
    - `infomax`: Very popular in EEGLAB (Matlab), good at finding "spiky" artifacts like blinks.
    - `picard`: A newer, more robsut version of *FASTICA*. It converges better (fails less often) and is generally recommended for modern analysis.<br />
    &emsp;&emsp; It further requires a *fine tune parameter* `fastica_it=n`, which tells the *Picard* algorithm: "Run n iterations of the *'FASTICA'* mode first to get a quick rough guess, then switch to the slower, more precise *'Picard'* mode to finish the job. This helps in speeding up the *Picard* process without losing accuracy.
+ `max_iter`: Since *ICA* is an iterative process - as it guesses the answer, checks if it's right, improves the guess, and repeats. This parameter sets the limit on how many times it can guess.
    > It recommended to set it to `500` or `1000` to ensure ICA runs until it finds the best possible solution, and if the limit is too low (e.g., 100) ICA might stop before it has fully separated the signals, giving you "mixed" components that are hard to interpret
+ `random_state=42`: Since, ICA starts with a random guess. Every time you it, the results might look slightly different (e.g., Component #1 today might be Component #5 tomorrow). Therefore, seeting a fixed seed ensures that every time we get the exact same results.
+ `fit_params=dict()`: Parameters specific to the `method` argument. For example, in the case of `method='picard'`, we can pass `fit_params=dict(fastica_it=5)` which tells the Picard algorithm: "Run 5 iterations of the 'FastICA' mode first to get a quick rough guess, then switc to the slower, more precise 'Picard' mode to finish the job".

In [ ]:
# initialise the ICA object
ica = mne.preprocessing.ICA(
    n_components=0.8,
    method='picard',
    max_iter=100,
    fit_params=dict(fastica_it=15),
    random_state=42
)

So, above we have told ICA to find enough independent sources to explain 80% (n_components) of my data using the smart Picard algorithm (method). I'm giving you 100 tries (max_iter) to get it right, but please start with 15 quick guesses (fit_params) to save time. And make sure you do it exactly the same way every time (random_state).
<br />
<br />
Now we can fit the new epochs by calling the `ICA.fit()` function and passing the new epochs to it:

In [ ]:
ica.fit(epochs_ica)

From the results above we can see *ICA* found 28 independent components to explain the 80% of total variance in the data.


To save the ICA object we can call its `.save()` function and to read ica object of the drive we can call the `mne.preprocessing.read_ica()` function:

In [ ]:
ica.save(
    fname="./out_data/epochs-ica.fif",
    overwrite=True
)

In [ ]:
ica = mne.preprocessing.read_ica(
    fname="./out_data/epochs-ica.fif"
)

ica

To view the independent components we can call the `.plot_componenets()` function of the `ICA` instance and also pass the `inst=orig_epochs` parameter as this will make the plot interactive: 

In [ ]:
ica.plot_components(inst=orig_epochs)

5. Now to detect which of these independent sources are artefacts or not we can use *MNE* help to do part of that work. To detect *ECG* and *EOG* patterns/artefacts:
    + We first have to create epochs each for the respective artefacts and luckily *MNE* comes with convenient fuctions for it `mne.preprocessing.create_ecg_epochs()` and `mne.preprocessing.create_eog_epochs()`. We have to feed them the orig raw data (i.e., not the raw where we perform a higher high pass filter of 1 Hz) and pass `tmin=-0.5`, `tmax=0.5`, `baseline=(None, -0.2)` and `reject=None` parameters specifications (these values tends to work better).
    > For example, in the case of *ECG*, *MNE* will try to figure out where the heart beats are and extract them as events and then based on these events create epochs

In [ ]:
# Let's first read the raw data with original filtering
bids_root = pathlib.Path("out_data/sample_BIDS")

bids_path = mne_bids.BIDSPath(
    subject='01',
    session='01',
    task='audiovisual',
    run='01',
    datatype='meg',
    root=bids_root
)

raw_bids = mne_bids.read_raw_bids(bids_path=bids_path)

# load the data in memory
raw_bids.load_data()
# perform band-pass filtering
raw_bids.filter(
    l_freq=0.1,
    h_freq=40.0
)

In [ ]:
ecg_epochs = mne.preprocessing.create_ecg_epochs(
    raw=raw_bids,
    reject=None,
    baseline=(None, -0.2),
    tmin=-0.5,
    tmax=0.5
)

eog_epochs = mne.preprocessing.create_eog_epochs(
    raw=raw_bids,
    reject=None,
    baseline=(None, -0.2),
    tmin=-0.5,
    tmax=0.5
)

+ Now, we are going to use the classification methods which comes with the *ICA* instances `ica.find_bad_ecg()` and `ica.find_bad_eog()` to classify which independent components are realted to the ecg and eog artefacts in thier respective epochs:

In [ ]:
ecg_bad_ic_inds, ecg_ic_scores = ica.find_bads_ecg(
    ecg_epochs,
    method='ctps' # classification method
)

eog_bad_ic_inds, eog_ic_scores = ica.find_bads_eog(
    eog_epochs
)

+ Now finally we are going to store these independent componenet indices that has been classified and pass them to the `.exclude` attribute of the *ICA*  instance:

In [ ]:
ic_to_exclude = ecg_bad_ic_inds + eog_bad_ic_inds
ica.exclude = ic_to_exclude

ica.exclude

+ Let's visualise the score ica gave to all the independent components using the `.plot_scores()` function of the *ICA* instance and see how extreme score the bad independent components have:

In [ ]:
ica.plot_scores(
    scores=ecg_ic_scores,
    title="ECG IC scores"
)

ica.plot_scores(
    scores=eog_ic_scores,
    title="EOG IC scores"
)

+ Using the `.plot_sources()` func of the *ICA* instance we can also have a look at the independent components sources . However for this we first have to create the evoked data of the artefact epochs:
> Note: Here we can see how the bad independent components in thier respective evoked artefacts are the extremes.

In [ ]:
ecg_evoked = ecg_epochs.average()

eog_evoked = eog_epochs.average()

In [ ]:
ica.plot_sources(
    inst = ecg_evoked,
    title = "ICA sources - ECG Evoked"    
)

ica.plot_sources(
    inst = eog_evoked,
    title = "ICA sources - EOG Evoked"
)

+ Using the same evoked data we can also use the `.plot_overlay()` of the *ICA* instance, to show what the data looks like after the removal of these ica bad independent components:

In [ ]:
ica.plot_overlay(
    inst = ecg_evoked,
    title = "ICA Overlay - ECG Evoked"
)

ica.plot_overlay(
    inst = eog_evoked,
    title = "ICA Overlay - EOG Evoked"
)

6. Finally the final step, cleaning our data by removing the bad independent components and for it all we need to do is call the `.apply()` func of the *ICA* instance and pass the **original epochs**:

In [ ]:
cleaned_orig_epochs = ica.apply(
    inst=orig_epochs.copy() # because it modifies the data inplace
)

Let's visualize the effect of ICA on the original epochs:

In [ ]:
cleaned_orig_epochs.plot(
    title="Cleaned Original Epochs - ICA Applied"
)
orig_epochs.plot(
    title="Original Epochs - Before ICA"
)

In [ ]:
cleaned_orig_epochs.save(
    "out_data/epochs-ica-cleaned-epo.fif"
)

## Artifact Correction Techniques Trade-Off
Even though *artifact correction techniques* like *SSP* and *ICA* can precisely remove artefacts from the recordings, but they can also inadvertently remove real brain activity and this is a fundamental trade-off of these techniques when preprocessing M/EEG data. <br />




### Why do SSP and ICA remove real brain activity?
The core reason is **Overlap**:

+ **SSP (Single Space Projection):** This method assumes that noise (like a heartbeat) lives in a different "spatial direction" than brain signal.
    - **The Problem:** If your brain signal (e.g., from the temporal lobe) happends to project in a similar direction as the heartbeat artifact, *SSP* will delete both. It's like using a color filter to remove "red" noise - if your picture has a red apple, the apple disappears too.
<div style="text-align: center;">
<img src="imgs/brain_lobes.png" alt="Different lobes of the brain with their characteristics." width="300">
<br />
*Diagram by: <a href="https://www.sciencedirect.com/topics/medicine-and-dentistry/temporal-lobe">sciencedirect</a>
</div>

+ **ICA (Independent Component Analysis):** This method separates signals based on "temporal independence" (different timing patters).
    - **The Problem:** If a brain source fires at the exact same time as an artifact (by random chance or systematic coupling), *ICA* might lump them into the same component. If you reject that component, you lose the brain signal

> Systematic coupling refers to the structured analysis of how different components, systems, or processes interact, influence, and depend on each other to achieve a coordinated, balanced outcome


### How do we tackle this?
We can use a combination of *visual checks (Topomap)* and *quantitative metrics (like SNR)* to ensure we are not losing real brain signals.
+ **Visual Topomap Inspection (The "Shape" check):**
    - Before applying an SSP projector or rejecting an ICA component, look at its Topomap.
    - *Artifact:* Distinct patterns (e.g., eyes = frontal lobe, heart = bottom-left dipole (left temporal lobe or left temporal tip)).
    - *Brain:* "Fuzzy" patterns or patterns centered over known sensory areas (visual/auditory cortex). If it looks like brain, keep it
<div style="text-align: center;">
<img src="imgs/pulse_artefact_topomap.png" alt="Pulse artefact topomap" width="500">
<br />
*Diagram by: <a href="https://pressrelease.brainproducts.com/eeg-artifacts-handling-in-analyzer/">brainproducts</a>
</div>

+ **Variance Explained (The "Strength" Check):**
    - *Red Flag:* If a single projector removes 20% of your data, it is likely eating brain signal. As, a typical heartbeat artifact might only be 1-5% of the total signal variance.

+ **SNR (Signal-to-Noise Ratio) Comparison:**
    - It is best to calculate the SNR before and after cleaning.
    - *Goal:* You want the SNR to increase.
    - *Warning:* If the SNR of your specific "target" signal (e.g., the N170 face response) dcreases after cleaning, you have over-cleaned the data.

### Do we also need to consider the experimental design?
The strategy significantly changes depending on your experimental design because the "nature" of the signal is different.

+ **Case A: Event-Related Data (e.g., Flashing lights, Beeeps)**
    -  *The Advantage:* You know exactly when the brain signal should happen (time-locked to the stimulus).
    - *The Strategy:*
        1. **Averaging is King:** Because you have many trails (e.g., 100 flashes), simply averaging them (computing the Evoked response) naturally cancels out random noise like heartbeats. You might not need aggressive SSP/ICA.
        2. **Epoch-Based Rejection:** Instead of trying to "fix" a bad blink with ICA, it's often safer to just throw away that single train (Epoch) entirely. You still have 99 other, but remember to be careful and check if not a good percentage of the data is removed.
        3. **Strict Filtering:** You can be more aggressive with filters (e.g., high-pass) because you care about short, sharp responses, not slow drifts.

+ **Case B: Resting-State Data (e.g., "Just sit and think")**
    - *The Challenge:* You don't know when the brain activity is happening. So, you can't average out trails to kill noise. Every second of data counts.
    - *The Strategy:*
        1. **Must Use ICA/SSP:** You cannot just "throw away" bad segments because you need long, continuous data for connectivity analysis. You must repair the artefacts using ICA or SSP.
        2. **Conservative Cleaning:** You have to be extremely careful. If you accidentally remove a "alpha wave" generator thinking it's noise, you have destroyed the primary result of your resting-state study.
        3. **No "Golden Standard":** Since you don't have a known "Evoked Response" to check against, you rely heavily on the Topomaps and spectral plots (frequenct content) to distinguish noise from brain.


### Summary
| Feature | Event-Related (ERP/ERF)| Resting-State | 
|---------|------------------------|---------------|
| *Primary Goal* | Clean the averaged response. | Clean the conttinuous raw signal. |
| *Reliance on Averaging* | High (Averaging kills most random noise). | None (Cannot average). |
| *Preferred Method* | Reject bad epochs (throw them away) | Repair artefacts (ICA/SSP). |
| *Risk of overr-cleaning* | Lower (Averaging protects the signal). | Highest (No ground truth to check against). |
| *SNR Check* | Check if the N100/P300 peak gets clearer. | Check if the Alpha/Beta peaks are preserved.|

>MNE provides an automated method to detect and annotate muscle artifacts using [`mne.preprocessing.annotate_muscle_zscore()`]((https://mne.tools/stable/auto_examples/preprocessing/muscle_detection.html)). By performing this step on the raw data before epoching, we ensure that any trial overlapping with a significant muscle artifact is automatically marked as "BAD" and dropped during the epoching stage (provided reject_by_annotation=True is set).
>
>This "reject-early" strategy is often preferred over trying to repair massive muscle artifacts later. While techniques like ICA can technically identify and remove muscle noise (since muscle tension often originates from spatially fixed sources), strong muscle artifacts are typically broadband and high-amplitude. They can overwhelm the ICA algorithm or require removing so many components that data quality suffers. Therefore, for event-related designs where we have plenty of trials, it is safer and cleaner to simply discard these contaminated epochs entirely rather than attempting to reconstruct them. However, in resting state, dropping segments creates "holes" in the continuous time series, which can mess up frequency analysis (PSDs) or connectivity measures. Therefore, in resting state, researchers often prefer ICA to repair the muscle artifacts so they can keep the continuous data intact.

# The End & Have a great day!